# Initialization Study — Stage 1

This notebook answers the first research question: how much classification
performance is surrendered by replacing Gaussian weight initialization with
generators an edge device can evaluate in native integer arithmetic.

The comparison is made on a **frozen** reservoir. Only the readout is trained, so
that no part of the difference between initializers can be attributed to
subsequent adaptation of the recurrent weights: what is measured is the quality
of the random projection itself, which is precisely the quantity the
initialization method determines. Stage 2 in Notebook 05 then asks how much of
any deficit training recovers.

Five methods are compared, all injected into an identical substrate at $k = T$
with the configuration selected in Notebook 03:

- **Gaussian** — the baseline, requiring a Box–Muller or inverse-CDF transform
- **LCG** and **LFSR** — pseudo-random generators built from an integer
  multiply-add and a shift-and-XOR respectively
- **Sobol** and **Halton** — deterministic low-discrepancy sequences

Every method is scaled to the same weight variance, $1/f_\text{eff}$. This is an
experimental control rather than a convenience: a uniform variate on $[-1,1]$ has
standard deviation $1/\sqrt{3}$ against the Gaussian's $1$, so without the
correction the baseline would carry weights $1.73$ times larger and any
difference would confound the *shape* of the distribution with its *scale*. The
final section reproduces that confound deliberately, as an ablation.

In [1]:
import pandas as pd
import torch

from edgesnn.data_utils import make_datasets
from edgesnn.experiments import (
    CONNECTIVITY, INIT_METHODS, N_RES, load_config, run_grid, seeds_for,
)
from edgesnn.initializers import initialize_reservoir, is_deterministic

DATA_DIR = "../Data"
DATASETS = ["ECG5000", "Wafer"]
N_SEEDS = 5

splits = {name: make_datasets(DATA_DIR, name) for name in DATASETS}
for method in INIT_METHODS:
    print(f"{method:<10} seeds={seeds_for(method, N_SEEDS)}"
          f"{'   (deterministic — one run, no error bar)' if is_deterministic(method) else ''}")

gaussian   seeds=[0, 1, 2, 3, 4]
lcg        seeds=[0, 1, 2, 3, 4]
lfsr       seeds=[0, 1, 2, 3, 4]
sobol      seeds=[0]   (deterministic — one run, no error bar)
halton     seeds=[0]   (deterministic — one run, no error bar)


## 1. Verifying the variance control

Before comparing the methods it is worth confirming that the control does what
it claims. Each generator is drawn at the reservoir's dimensions and its
empirical standard deviation compared against the target $1/\sqrt{f_\text{eff}}$,
where the effective fan-in counts only the synapses that exist.

If the five agree, then any difference in the results that follow is a
consequence of how the weights are *distributed*, not of how large they are.

In [2]:
target = (CONNECTIVITY * N_RES) ** -0.5
rows = []
for method in INIT_METHODS:
    weights = initialize_reservoir((N_RES, N_RES), method, fan_in=N_RES,
                                   connectivity=CONNECTIVITY, seed=0)
    active = weights[weights != 0]
    rows.append({"method": method, "std": active.std().item(),
                 "target": target,
                 "deviation": active.std().item() / target - 1,
                 "non_zeros": int((weights != 0).sum())})
display(pd.DataFrame(rows).round(4))

,method,std,target,deviation,non_zeros
0,gaussian,0.1962,0.1976,-0.0075,6554
1,lcg,0.1964,0.1976,-0.0064,6554
2,lfsr,0.1980,0.1976,0.0016,6554
3,sobol,0.1981,0.1976,0.0023,6554
4,halton,0.1995,0.1976,0.0095,6554


The non-zero count is identical across methods by construction: the connectivity
mask fixes an exact number of synapses rather than drawing each entry
independently, so every initializer operates under the same synaptic budget. That
matters for the energy comparison, since the synapse count is exactly the term
converting spikes into synaptic operations.

## 2. Determinism of the low-discrepancy sequences

Sobol and Halton are constructions, not samplers: the $n$-th point is a fixed
function of $n$, so the seed has nothing to act upon. The check below confirms
this holds in the implementation, because it determines how the two are run —
once each, with no error bar — and reporting a standard deviation of zero is only
honest if it reflects the mathematics rather than an oversight.

In [3]:
for method in INIT_METHODS:
    a = initialize_reservoir((N_RES, N_RES), method, fan_in=N_RES, seed=0)
    b = initialize_reservoir((N_RES, N_RES), method, fan_in=N_RES, seed=99)
    identical = torch.equal(a, b)
    print(f"{method:<10} seed 0 vs seed 99: "
          f"{'IDENTICAL — seed has no effect' if identical else 'differs'}")

gaussian   seed 0 vs seed 99: differs
lcg        seed 0 vs seed 99: differs
lfsr       seed 0 vs seed 99: differs
sobol      seed 0 vs seed 99: IDENTICAL — seed has no effect
halton     seed 0 vs seed 99: IDENTICAL — seed has no effect


## 3. Stage 1: frozen reservoir, five initializers

Each method is injected into the frozen substrate and only the readout is
trained, at $k = T$. The stochastic methods are repeated over five seeds; the
deterministic ones are run once, as established above.

Results are appended to disk as each condition completes, so an interrupted run
resumes rather than restarting.

In [4]:
stage1 = {}

for name, (train_ds, test_ds) in splits.items():
    config = load_config(f"config_lsm_{name}")
    if config is None:
        raise RuntimeError(f"Run notebook 03 first — no tuned configuration for {name}.")

    conditions = [
        dict(kind="lsm", k=train_ds.seq_len, init_method=method, seed=seed,
             config=dict(config))
        for method in INIT_METHODS
        for seed in seeds_for(method, N_SEEDS)
    ]
    stage1[name] = run_grid(f"stage1_init_{name}", conditions, train_ds, test_ds,
                            max_epochs=80, min_epochs=15, patience=12)

stage1_init_ECG5000: 17 conditions to run


stage1_init_ECG5000:   0%|          | 0/17 [00:00<?, ?run/s]

stage1_init_Wafer: 17 conditions to run


stage1_init_Wafer:   0%|          | 0/17 [00:00<?, ?run/s]

In [5]:
tables = {}
for name, frame in stage1.items():
    table = (frame.groupby("init_method")
             .agg(macro_f1=("macro_f1", "mean"), sd=("macro_f1", "std"),
                  accuracy=("accuracy", "mean"), runs=("macro_f1", "size"),
                  firing_rate=("firing_rate", "mean"), sops=("sops", "mean"))
             .reindex(INIT_METHODS))
    table["sd"] = table["sd"].fillna(0.0)      # a single deterministic run
    table["vs_gaussian"] = table.macro_f1 - table.loc["gaussian", "macro_f1"]
    tables[name] = table
    print(f"\n{name} — frozen reservoir, k = T")
    display(table.round(4))


ECG5000 — frozen reservoir, k = T


,macro_f1,sd,accuracy,runs,firing_rate,sops,vs_gaussian
init_method,,,,,,,
gaussian,0.9433,0.0179,0.9456,5,0.3621,332023.2500,0.0000
lcg,0.9382,0.0110,0.9403,5,0.3625,333044.7625,-0.0051
lfsr,0.9388,0.0253,0.9412,5,0.3819,349516.9500,-0.0044
sobol,0.9349,0.0000,0.9376,1,0.4223,391060.8125,-0.0084
halton,0.9346,0.0000,0.9373,1,0.4190,383601.6250,-0.0087



Wafer — frozen reservoir, k = T


,macro_f1,sd,accuracy,runs,firing_rate,sops,vs_gaussian
init_method,,,,,,,
gaussian,0.9027,0.0065,0.9645,5,0.3707,369196.9312,0.0000
lcg,0.8816,0.0239,0.9560,5,0.3755,374164.0875,-0.0212
lfsr,0.8699,0.0314,0.9531,5,0.3923,389838.1062,-0.0328
sobol,0.7694,0.0000,0.9095,1,0.4616,463740.3750,-0.1333
halton,0.7371,0.0000,0.9215,1,0.4507,448122.7812,-0.1656


The `vs_gaussian` column is the answer to the first research question, stated per
dataset: the performance given up, or gained, by abandoning the Gaussian
transform. It should be read against the `sd` column, since a deficit smaller
than the seed-to-seed spread is not evidence of a difference between methods.

The `firing_rate` and `sops` columns matter independently of accuracy. Two
initializers that classify equally well but drive the reservoir at different
rates do not cost the same to run, and on an edge device that difference is as
consequential as the accuracy.

## 4. Ablation: removing the variance control

The comparison above equalises the weight variance across methods. This section
removes that correction, so that the uniform generators keep their natural
standard deviation of $1/\sqrt{3}$ while the Gaussian baseline retains $1$.

The purpose is to measure the size of the confound the control removes. If the
ranking of methods changes when the correction is withdrawn, then a study run
without it would have been measuring scale while claiming to measure
distribution shape.

In [6]:
ablation = {}

for name, (train_ds, test_ds) in splits.items():
    config = load_config(f"config_lsm_{name}")
    conditions = [
        dict(kind="lsm", k=train_ds.seq_len, init_method=method, seed=seed,
             config=dict(config), match_variance=False)
        for method in INIT_METHODS
        for seed in seeds_for(method, N_SEEDS)
    ]
    ablation[name] = run_grid(f"stage1_ablation_{name}", conditions,
                              train_ds, test_ds, max_epochs=80, min_epochs=15,
                              patience=12)

stage1_ablation_ECG5000: 17 conditions to run


stage1_ablation_ECG5000:   0%|          | 0/17 [00:00<?, ?run/s]

stage1_ablation_Wafer: 17 conditions to run


stage1_ablation_Wafer:   0%|          | 0/17 [00:00<?, ?run/s]

In [7]:
for name in splits:
    controlled = tables[name][["macro_f1"]].rename(columns={"macro_f1": "with control"})
    uncontrolled = (ablation[name].groupby("init_method").macro_f1.mean()
                    .reindex(INIT_METHODS).rename("without control"))
    comparison = controlled.join(uncontrolled)
    comparison["difference"] = comparison["without control"] - comparison["with control"]
    print(f"\n{name} — effect of the variance control")
    display(comparison.round(4))


ECG5000 — effect of the variance control


,with control,without control,difference
init_method,,,
gaussian,0.9433,0.9433,0.0000
lcg,0.9382,0.9308,-0.0074
lfsr,0.9388,0.9338,-0.0050
sobol,0.9349,0.9420,0.0071
halton,0.9346,0.8997,-0.0349



Wafer — effect of the variance control


,with control,without control,difference
init_method,,,
gaussian,0.9027,0.9027,0.0000
lcg,0.8816,0.7616,-0.1199
lfsr,0.8699,0.7589,-0.1110
sobol,0.7694,0.6159,-0.1535
halton,0.7371,0.7132,-0.0240


## 5. What this establishes

Stage 1 fixes the cost of cheap randomness on a frozen substrate, which is the
pessimistic case: the reservoir has no opportunity to compensate for whatever the
initializer got wrong. Whichever cheap method comes closest to the Gaussian
baseline here is carried into Stage 2, where truncated surrogate-gradient
training is allowed to adapt the reservoir and the question becomes how much of
the remaining deficit that training recovers.

The deterministic methods deserve separate comment in the results. Their zero
variance is not a small standard deviation but an exact one, and for an edge
deployment it is a property worth having in its own right: a device using a Sobol
or Halton reservoir needs neither a stored seed nor an entropy source, and
reproduces the identical network on every power-on.